# DNA Ingestion: Promoter vs Non-Promoter (Ensembl GRCh38)

This notebook builds the DNA classification dataset for the locked DNA task.

**Task:** promoter vs non-promoter (binary)  
**Organism:** human (GRCh38)  
**Sequence length:** fixed (from `configs/config.yaml`)  
**Sources:**
- Ensembl FTP: GTF annotation (gene coordinates)
- Ensembl REST: chromosome lengths + sequence retrieval by region

## Outputs
- Cached GTF: `data/raw/Homo_sapiens.GRCh38.*.gtf.gz`
- Promoter candidates: `data/raw/dna_promoter_candidates_len{L}.csv`
- Negative candidates: `data/raw/dna_negative_candidates_len{L}.csv`
- Final dataset: `data/processed/dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv`
- Ingestion summary: `reports/dna_ingest_summary.json`

## Label definition (prototype)
- **Positive (label=1):** fixed-length window centered at gene TSS
- **Negative (label=0):** random genomic windows on standard chromosomes, filtered to avoid promoter overlap

In [1]:
from pathlib import Path
import gzip
import json
import random
import re
import time
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import requests
import yaml
from tqdm.auto import tqdm

/opt/anaconda3/envs/bioseq-capstone/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Paths + config

This notebook assumes it is located in `notebooks/` so `ROOT = parent of current working directory`.

In [2]:
ROOT = Path.cwd().parents[0]
DATA = ROOT / "data"
RAW = DATA / "raw"
PROCESSED = DATA / "processed"
REPORTS = ROOT / "reports"
CONFIGS = ROOT / "configs"

for p in [RAW, PROCESSED, REPORTS]:
    p.mkdir(parents=True, exist_ok=True)

cfg_path = CONFIGS / "config.yaml"
assert cfg_path.exists(), f"Missing config: {cfg_path}"

with open(cfg_path, "r") as f:
    cfg = yaml.safe_load(f)

SEED = int(cfg["project"]["random_seed"])
ORG = cfg["dna"]["organism"]          # "human"
BUILD = cfg["dna"]["genome_build"]    # "GRCh38"
L = int(cfg["dna"]["seq_length_bp"])
N_POS = int(cfg["dna"]["n_pos"])
N_NEG = int(cfg["dna"]["n_neg"])

random.seed(SEED)
np.random.seed(SEED)

(SEED, ORG, BUILD, L, N_POS, N_NEG)

(42, 'human', 'GRCh38', 200, 2000, 2000)

## Ensembl endpoints + helpers

We use:
- `/info/assembly/homo_sapiens` to get chromosome lengths (for bounds + negative sampling)
- `/sequence/region/human/{region}` to fetch DNA sequences

We add simple retry/backoff for HTTP 429 rate-limiting.

In [3]:
ENSEMBL_REST = "https://rest.ensembl.org"

def ensembl_get_json(url: str, params=None, max_tries: int = 6, sleep: float = 0.6):
    headers = {"Content-Type": "application/json"}
    last_err = None
    for t in range(max_tries):
        r = requests.get(url, headers=headers, params=params, timeout=90)
        if r.status_code == 429:
            time.sleep(sleep * (t + 1))
            continue
        try:
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last_err = e
            time.sleep(sleep * (t + 1))
    raise RuntimeError(f"Failed after {max_tries} tries: {url}. Last error: {last_err}")

def ensembl_get_text(url: str, params=None, max_tries: int = 6, sleep: float = 0.6):
    headers = {"Content-Type": "text/plain"}
    last_err = None
    for t in range(max_tries):
        r = requests.get(url, headers=headers, params=params, timeout=90)
        if r.status_code == 429:
            time.sleep(sleep * (t + 1))
            continue
        try:
            r.raise_for_status()
            return r.text
        except Exception as e:
            last_err = e
            time.sleep(sleep * (t + 1))
    raise RuntimeError(f"Failed after {max_tries} tries: {url}. Last error: {last_err}")

## Fetch chromosome lengths (standard chromosomes only)

We keep chromosomes {1..22, X, Y} for a clean prototype.

In [4]:
assembly = ensembl_get_json(f"{ENSEMBL_REST}/info/assembly/homo_sapiens")
top = assembly.get("top_level_region", [])

allowed = {str(i) for i in range(1, 23)} | {"X", "Y"}
chrom_sizes = {r["name"]: int(r["length"]) for r in top if r.get("name") in allowed}

assert len(chrom_sizes) == 24, f"Expected 24 standard chromosomes, got {len(chrom_sizes)}"
list(chrom_sizes.items())[:5]

[('Y', 57227415),
 ('20', 64444167),
 ('X', 156040895),
 ('13', 114364328),
 ('22', 50818468)]

## Download the correct Ensembl GTF (GRCh38) if missing

Important: avoid `abinitio` GTFs.  
We select a GRCh38 GTF from the `current_gtf` directory and prefer the canonical file (non-abinitio).

In [5]:
GTF_DIR = "https://ftp.ensembl.org/pub/current_gtf/homo_sapiens/"

html = requests.get(GTF_DIR, timeout=60).text
all_gtfs = re.findall(r'href="(Homo_sapiens\.GRCh38\.[^"]+\.gtf\.gz)"', html)
assert all_gtfs, "No GRCh38 GTFs found in Ensembl FTP listing."

preferred = [x for x in all_gtfs if ("abinitio" not in x and "chr_patch_hapl_scaff" not in x)]
if not preferred:
    preferred = [x for x in all_gtfs if "abinitio" not in x]
assert preferred, f"Only abinitio GTFs found. Candidates: {all_gtfs[:10]}"

gtf_name = sorted(preferred, key=len)[0]
gtf_url = GTF_DIR + gtf_name
gtf_path = RAW / gtf_name

gtf_name, gtf_path

('Homo_sapiens.GRCh38.115.gtf.gz',
 PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/raw/Homo_sapiens.GRCh38.115.gtf.gz'))

In [6]:
if not gtf_path.exists():
    print("Downloading:", gtf_url)
    r = requests.get(gtf_url, stream=True, timeout=180)
    r.raise_for_status()
    with open(gtf_path, "wb") as f:
        for chunk in r.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
else:
    print("Using cached:", gtf_path)

gtf_path.exists(), gtf_path.stat().st_size

Using cached: /Users/saturnine/Projects/bio-seq-lm-capstone/data/raw/Homo_sapiens.GRCh38.115.gtf.gz


(True, 104395529)

## Parse genes from GTF and build promoter windows

Definition (fixed length `L`):
- TSS:
  - `+` strand: TSS = start
  - `-` strand: TSS = end
- Promoter window centered at TSS:
  - `start = TSS - L//2`
  - `end = start + L - 1`

Filters:
- Chromosome in {1..22, X, Y}
- Valid genomic bounds: `start >= 1` and `end <= chrom_length`
- Feature must be `gene`

In [7]:
def parse_gtf_attributes(attr_str: str) -> dict:
    attrs = {}
    for part in attr_str.strip().split(";"):
        part = part.strip()
        if not part:
            continue
        m = re.match(r'(\S+)\s+"(.+)"', part)
        if m:
            attrs[m.group(1)] = m.group(2)
    return attrs

rows = []
with gzip.open(gtf_path, "rt") as f:
    for line in f:
        if line.startswith("#"):
            continue
        parts = line.rstrip("\n").split("\t")
        if len(parts) != 9:
            continue

        chrom, source, feature, start, end, score, strand, frame, attrs = parts
        chrom = chrom.replace("chr", "")  # safety

        if feature != "gene":
            continue
        if chrom not in chrom_sizes:
            continue
        if strand not in {"+", "-"}:
            continue

        start = int(start)
        end = int(end)

        ad = parse_gtf_attributes(attrs)
        gene_id = ad.get("gene_id")
        gene_name = ad.get("gene_name", "")
        gene_biotype = ad.get("gene_biotype", ad.get("gene_type", ""))

        # skip if no gene_id
        if not gene_id:
            continue

        tss = start if strand == "+" else end
        prom_start = tss - (L // 2)
        prom_end = prom_start + L - 1

        if prom_start < 1:
            continue
        if prom_end > chrom_sizes[chrom]:
            continue

        rows.append((gene_id, gene_name, gene_biotype, chrom, strand, tss, prom_start, prom_end))

genes = pd.DataFrame(
    rows,
    columns=["gene_id", "gene_name", "gene_biotype", "chrom", "strand", "tss", "start", "end"]
)

genes.shape, genes.head(3)

((78654, 8),
            gene_id gene_name    gene_biotype chrom strand      tss    start  \
 0  ENSG00000142611    PRDM16  protein_coding     1      +  3069168  3069068   
 1  ENSG00000284616                    lncRNA     1      -  5307394  5307294   
 2  ENSG00000260972                    lncRNA     1      +  5492978  5492878   
 
        end  
 0  3069267  
 1  5307493  
 2  5493077  )

### Quick sanity check

We want a non-zero number of gene-based promoter candidates.  
If this is still zero, it means the GTF selection is wrong (abinitio or unexpected format).

In [8]:
assert genes.shape[0] > 0, (
    "No gene rows parsed. This usually means the wrong GTF was downloaded "
    "(e.g., abinitio). Check gtf_name and re-run the download cell."
)

genes["chrom"].value_counts().head()

chrom
1     7090
2     5686
6     4230
11    4166
3     4158
Name: count, dtype: int64

## Sample positive promoters (N_POS)

We sample fixed-length promoter windows from the gene list with a fixed seed.

In [9]:
assert genes.shape[0] >= N_POS, f"Not enough promoter candidates ({genes.shape[0]}) for N_POS={N_POS}"

pos = genes.sample(n=N_POS, random_state=SEED).reset_index(drop=True).copy()
pos["label"] = 1
pos["region_id"] = [f"prom_{i}" for i in range(len(pos))]

pos_out = RAW / f"dna_promoter_candidates_len{L}_pos{N_POS}.csv"
pos.to_csv(pos_out, index=False)

pos_out, pos.shape

(PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/raw/dna_promoter_candidates_len200_pos2000.csv'),
 (2000, 10))

## Negative sampling (N_NEG)

We generate random genomic windows of length `L` on standard chromosomes.

Constraints:
- Valid bounds: `1 <= start` and `end <= chrom_length`
- Avoid overlap with any promoter window **on the same chromosome**

Overlap rule:
- A negative window is rejected if it intersects any promoter interval on that chromosome.

This is a simple prototype approach and keeps scope tight.

In [10]:
# Build promoter intervals per chromosome for fast overlap checks
promoters_by_chrom: Dict[str, List[Tuple[int, int]]] = {}
for chrom, sub in genes.groupby("chrom"):
    intervals = list(zip(sub["start"].astype(int).tolist(), sub["end"].astype(int).tolist()))
    intervals.sort(key=lambda x: x[0])
    promoters_by_chrom[chrom] = intervals

def overlaps_any(intervals: List[Tuple[int, int]], s: int, e: int) -> bool:
    # Linear scan is ok for prototype sizes; could be optimized later if needed.
    for a, b in intervals:
        if b < s:
            continue
        if a > e:
            break
        return True
    return False

In [11]:
chrom_list = sorted(chrom_sizes.keys(), key=lambda x: (len(x), x))  # stable order

neg_rows = []
tries = 0
max_tries = N_NEG * 50  # guardrail

pbar = tqdm(total=N_NEG, desc="Sampling negatives")
while len(neg_rows) < N_NEG and tries < max_tries:
    tries += 1
    chrom = random.choice(chrom_list)
    chrom_len = chrom_sizes[chrom]

    # choose start so that end <= chrom_len
    start = random.randint(1, chrom_len - L + 1)
    end = start + L - 1

    # reject if overlaps any promoter on that chromosome
    if overlaps_any(promoters_by_chrom.get(chrom, []), start, end):
        continue

    neg_rows.append(("", "", "", chrom, ".", -1, start, end))  # gene placeholders
    pbar.update(1)

pbar.close()

assert len(neg_rows) == N_NEG, f"Could only sample {len(neg_rows)}/{N_NEG} negatives (tries={tries})."

neg = pd.DataFrame(
    neg_rows,
    columns=["gene_id", "gene_name", "gene_biotype", "chrom", "strand", "tss", "start", "end"]
)
neg["label"] = 0
neg["region_id"] = [f"neg_{i}" for i in range(len(neg))]

neg_out = RAW / f"dna_negative_candidates_len{L}_neg{N_NEG}.csv"
neg.to_csv(neg_out, index=False)

neg_out, neg.shape

Sampling negatives: 100%|████████████████| 2000/2000 [00:00<00:00, 24534.19it/s]


(PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/raw/dna_negative_candidates_len200_neg2000.csv'),
 (2000, 10))

## Fetch DNA sequences from Ensembl REST

We fetch sequences for regions like:
`{chrom}:{start}..{end}`

Notes:
- Uses `human` in the REST path.
- We request plain text and uppercase it.
- We do minimal validation: sequence length == L and only contains A/C/G/T (allow N if it appears).

In [12]:
def fetch_region_seq(chrom: str, start: int, end: int) -> str:
    region = f"{chrom}:{start}..{end}"
    url = f"{ENSEMBL_REST}/sequence/region/{ORG}/{region}"
    seq = ensembl_get_text(url, params=None).strip().upper()
    return seq

In [13]:
def fetch_sequences(df_regions: pd.DataFrame, label_name: str) -> pd.DataFrame:
    seqs = []
    for _, row in tqdm(df_regions.iterrows(), total=len(df_regions), desc=f"Fetching {label_name} seqs"):
        chrom = row["chrom"]
        start = int(row["start"])
        end = int(row["end"])
        seq = fetch_region_seq(chrom, start, end)
        seqs.append(seq)
    out = df_regions.copy()
    out["sequence"] = seqs
    return out

pos_seq = fetch_sequences(pos[["region_id", "chrom", "start", "end", "label"]], "positive")
neg_seq = fetch_sequences(neg[["region_id", "chrom", "start", "end", "label"]], "negative")

pos_seq.head(2), neg_seq.head(2)

Fetching negative seqs: 100%|███████████████| 2000/2000 [19:57<00:00,  1.67it/s]


(  region_id chrom      start        end  label  \
 0    prom_0     4   88592334   88592533      1   
 1    prom_1     5  120078977  120079176      1   
 
                                             sequence  
 0  TGCCCGCGGACCTTGCCGCCCCGCCTCCAGCCCGTGCCACGGCGGC...  
 1  AGAAAGAAGGAGAAAAAACGGCTCAAAGAAGAGTTGATGGCTGGGA...  ,
   region_id chrom      start        end  label  \
 0     neg_0    19    7471302    7471501      0   
 1     neg_1     1  199058448  199058647      0   
 
                                             sequence  
 0  GCCCCTCGCTTGTAGTGAGCTGTTGCAGCTTACGGTCCGTTCCCTG...  
 1  TGGCTCAGTGATCTCATCTTTTCTTTGATCAGGACATGGTGACAGT...  )

## Combine + validate + save processed dataset

In [14]:
dna = pd.concat([pos_seq, neg_seq], ignore_index=True)

# Basic validations
dna["seq_len"] = dna["sequence"].str.len()
bad_len = int((dna["seq_len"] != L).sum())

# allow A/C/G/T/N only (N can appear in assembly gaps)
allowed_chars = set("ACGTN")
bad_chars = int((dna["sequence"].apply(lambda s: any(ch not in allowed_chars for ch in s))).sum())

bad_len, bad_chars, dna.shape

(0, 0, (4000, 7))

In [15]:
assert bad_len == 0, f"Found {bad_len} sequences not matching expected length L={L}"
assert bad_chars == 0, f"Found {bad_chars} sequences with non-ACGTN characters"

out_csv = PROCESSED / f"dna_promoter_vs_nonpromoter_len{L}_pos{N_POS}_neg{N_NEG}.csv"
dna_out = dna[["region_id", "chrom", "start", "end", "sequence", "label"]].copy()
dna_out.to_csv(out_csv, index=False)

out_csv, dna_out.shape

(PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/data/processed/dna_promoter_vs_nonpromoter_len200_pos2000_neg2000.csv'),
 (4000, 6))

## Save ingestion summary

In [16]:
summary = {
    "gtf_file": str(gtf_path),
    "gtf_name": gtf_name,
    "organism": ORG,
    "genome_build": BUILD,
    "seq_length_bp": int(L),
    "n_pos": int(N_POS),
    "n_neg": int(N_NEG),
    "pos_candidates_total": int(genes.shape[0]),
    "neg_sampling_tries": int(tries),
    "outputs": {
        "pos_candidates_csv": str(pos_out),
        "neg_candidates_csv": str(neg_out),
        "processed_dataset_csv": str(out_csv),
    },
    "validation": {
        "bad_length": int(bad_len),
        "bad_characters": int(bad_chars),
        "allowed_characters": "ACGTN",
    },
    "seed": int(SEED),
}

summary_path = REPORTS / "dna_ingest_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)

summary_path

PosixPath('/Users/saturnine/Projects/bio-seq-lm-capstone/reports/dna_ingest_summary.json')